# CrossRef Reference Verifier

Advancing AI Anthropology through computational approaches to qualitative research.

[Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

<br>

---

<br>

## What This Notebook Does

This notebook verifies reference lists against the CrossRef registry — the canonical record of scholarly metadata. Paste a paper's reference section and it resolves every DOI, compares the citation as written against the canonical record field by field (title, first author, year, journal), flags mismatches and retractions, and produces corrected citations. References without DOIs are matched against CrossRef's bibliographic search to recover the missing identifier — or to reveal that no plausible match exists, which is how fabricated references announce themselves.

The notebook also queries CrossRef directly: all works from a journal by ISSN (with date ranges), and all works by an author via ORCID. Everything runs against the free public API — no key, no scraping, no LLM.

**A note on what is sent where:** reference text is sent to the public CrossRef API to be matched. Reference lists are published metadata, not participant data, but the disclosure is worth knowing.

## Key Features

- **Reference verification**: Per-entry verdicts — OK, mismatch (with field-level detail), DOI invalid, or needs review — comparing citation text against canonical CrossRef metadata
- **Hallucination detection**: Entries without DOIs are matched bibliographically; a reference that matches nothing with confidence is a reference to investigate
- **Retraction and correction flags**: Each resolved DOI is checked for retraction, correction, and erratum notices
- **Corrected citations**: Canonical metadata rendered as clean reference text and BibTeX
- **Journal query**: All works from any journal by ISSN, with date filters and pagination
- **Author query**: All works linked to an ORCID iD
- **Multi-format export**: CSV, styled Excel, JSON, and BibTeX

## Workflow

1. **Setup**: Install packages and load utilities
2. **Configure**: Set your polite-pool email and request pacing
3. **Verify**: Paste a reference list and run the verification report
4. **Query**: Pull journal (ISSN) or author (ORCID) records as needed
5. **Export**: Download reports and datasets in your preferred format

## Applications

This notebook supports citation integrity across the research lifecycle: checking a manuscript's references before submission, auditing an AI-assisted literature review for fabricated citations, verifying a syllabus or bibliography, monitoring retractions among load-bearing sources, and building journal or author publication datasets for bibliometric work. It pairs with the toolkit's literature-review skill, whose first guardrail — never fabricate citations, verify DOIs resolve — this notebook operationalizes.

## Methodological Positioning

This notebook represents a **computational anthropology** approach to a problem the AI era has sharpened: reference lists now fail in new ways, from language-model hallucination to silent metadata drift. Verification against the canonical registry is a mechanical task machines do well, so the researcher's attention can go where judgment is required. The comparison logic deliberately flags rather than auto-judges uncertain cases — journal abbreviations, translated titles, and variant author spellings need a human reader, and the report marks them for review instead of guessing.

## Target Audience

Designed for researchers, editors, peer reviewers, and students who need reference lists they can trust — no programming experience required.

## Technical Approach

The notebook uses the public CrossRef REST API with polite-pool identification (your email in the User-Agent). Verification compares citation text to canonical metadata using conservative token-based matching; bibliographic recovery uses CrossRef's `query.bibliographic` scored search; retraction signals come from CrossRef's update relationships, which carry Retraction Watch data. Journal and author queries use cursor pagination. All processing beyond the API calls is local.

## AI Anthropology Toolkit

This notebook is part of the [AI Anthropology Toolkit](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit), a collection of computational notebooks for AI-assisted qualitative research. It pairs with the [Academic Literature Explorer](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit/blob/main/notebooks/Academic_Literature_Explorer.ipynb) for discovery and with the toolkit's literature-review skill for screening and synthesis.

## License & Attribution

This notebook is released under the [PolyForm Noncommercial License 1.0.0](https://polyformproject.org/licenses/noncommercial/1.0.0): free for noncommercial use — research, education, and nonprofit work — with attribution to Matt Artz appreciated. For commercial licensing, contact [Matt Artz](https://www.mattartz.me/).

## Citation

If you use this toolkit in your academic research, please cite:

> Artz, Matt. 2026. AI Anthropology Toolkit. Software. Zenodo. https://doi.org/10.5281/zenodo.16728812

## References

Artz, Matt. 2023. From Machine Learning to Machine Knowing: A Digital Anthropology Approach for the Machine Interpretation of Cultures. UNESCO. https://unesdoc.unesco.org/ark:/48223/pf0000384902.

Artz, Matt. 2023. "Ten Predictions for AI and the Future of Anthropology." Anthropology News, May 8. https://doi.org/10.1111/AN.1605.

Artz, Matt. 2026. "Artificial Intelligence: The AI Anthropology Lifecycle (of, by, for AI)." In Practicing Digital Ethnography, edited by Devin Proctor. Routledge. https://doi.org/10.4324/9781032672663-29.

Artz, Matt. 2026. "Multi-Agent Ethnography: Post-Conventional Anthropological Practice Through Human-AI Collaboration." Anthropological Forum. https://doi.org/10.1080/00664677.2026.2614501.

Artz, Matt. Forthcoming. "AI Anthropology: The Future of Applied Anthropological Practice." In Routledge Handbook of Applied Anthropology, edited by Christina Wasson, Edward B. Liebow, Karine L. Narahara, Ndukuyakhe Ndlovu, and Alaka Wali. New York: Routledge.


## Setup and Installation

Install the required packages and load the verification utilities. Everything uses the free public CrossRef API — no account or key required.

In [ ]:
# Install required packages (numpy/pandas pinned for Colab compatibility)
!pip install -q requests pandas "numpy<2.2" openpyxl python-docx ipywidgets

import re
import json
import time
import unicodedata
import requests
import pandas as pd
from datetime import datetime
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# python-docx for .docx reference uploads
import docx
print("✓ python-docx loaded")

# Colab detection
IN_COLAB = False
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    pass

# --- Utilities ---

def make_slug(text):
    """Filesystem-safe slug from text."""
    text = unicodedata.normalize('NFKD', text)
    text = re.sub(r'[^\w\s-]', '', text).strip().lower()
    return re.sub(r'[-\s]+', '_', text)[:50] or 'project'

def normalize_text(text):
    """Normalize unicode artifacts."""
    if not isinstance(text, str):
        return text
    replacements = {
        '‑': '-', '‐': '-', '‒': '-', '–': '-', '—': '-',
        '‘': "'", '’': "'", '“': '"', '”': '"',
        '…': '...', '\xa0': ' ',
    }
    for k, v in replacements.items():
        text = text.replace(k, v)
    return text

def style_excel(filepath):
    """Apply branded styling to an Excel file."""
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    wb = load_workbook(filepath)
    hf = PatternFill(start_color='274C77', end_color='274C77', fill_type='solid')
    hfont = Font(bold=True, color='FFFFFF')
    tb = Border(
        left=Side(style='thin', color='E7ECEF'), right=Side(style='thin', color='E7ECEF'),
        top=Side(style='thin', color='E7ECEF'), bottom=Side(style='thin', color='E7ECEF'))
    for ws in wb.worksheets:
        for cell in ws[1]:
            cell.fill = hf
            cell.font = hfont
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border = tb
        for col in ws.columns:
            max_len = max(len(str(c.value or '')) for c in col)
            ws.column_dimensions[col[0].column_letter].width = min(max(max_len + 2, 10), 60)
        ws.freeze_panes = 'A2'
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(vertical='top', wrap_text=True)
                cell.border = tb
    wb.save(filepath)

# --- DOI and reference parsing ---

DOI_PATTERN = re.compile(r'10\.\d{4,9}/[^\s"<>]+', re.IGNORECASE)

def clean_doi(doi_input):
    """Normalize a DOI to bare form (10.xxxx/yyyy); None if invalid."""
    if not doi_input or not isinstance(doi_input, str):
        return None
    s = str(doi_input).strip()
    for pattern in (r'https?://(?:dx\.)?doi\.org/(.+)', r'doi:\s*(.+)'):
        m = re.search(pattern, s, re.IGNORECASE)
        if m:
            s = m.group(1)
            break
    m = DOI_PATTERN.search(s)
    if not m:
        return None
    doi = m.group(0).rstrip('.,;)]}"\'')
    return doi if re.match(r'^10\.\d{4,9}/.+', doi) else None

def clean_orcid(orcid_input):
    """Normalize an ORCID iD to bare form (0000-0000-0000-0000)."""
    if not orcid_input or not isinstance(orcid_input, str):
        return None
    m = re.search(r'(\d{4}-\d{4}-\d{4}-\d{3}[\dX])', str(orcid_input), re.IGNORECASE)
    return m.group(1).upper() if m else None

def split_reference_entries(text):
    """Split a pasted reference section into individual entries.

    Handles blank-line-separated blocks and one-entry-per-line lists;
    continuation lines (starting with whitespace) join the previous entry."""
    text = normalize_text(text).strip()
    if not text:
        return []
    blocks = [b.strip() for b in re.split(r'\n\s*\n', text) if b.strip()]
    if len(blocks) > 1:
        return [re.sub(r'\s+', ' ', b) for b in blocks]
    entries = []
    for line in text.splitlines():
        if not line.strip():
            continue
        if line[:1].isspace() and entries:
            entries[-1] += ' ' + line.strip()
        else:
            entries.append(line.strip())
    return [re.sub(r'\s+', ' ', e) for e in entries if len(e.strip()) >= 20]

# --- Comparison helpers (conservative: flag, don't auto-judge) ---

_STOP = {'a', 'an', 'and', 'the', 'of', 'in', 'on', 'for', 'to', 'with'}

def norm_tokens(s):
    return [t for t in re.findall(r'[a-z0-9]+', (s or '').lower()) if t not in _STOP]

def title_similarity(citation_text, canonical_title):
    """Fraction of canonical-title tokens present in the citation text."""
    ct = set(norm_tokens(citation_text))
    tt = norm_tokens(canonical_title)
    if not tt:
        return 0.0
    return sum(1 for t in tt if t in ct) / len(tt)

def container_check(citation_text, canonical_container, canonical_title=''):
    """'pass' | 'review' | 'fail' for the journal/container name.

    Judged primarily on container tokens NOT shared with the work's title,
    so a title word like "Anthropological" cannot vouch for a journal name.
    Abbreviations (Am. Ethnol.) can't be auto-matched reliably, so partial
    evidence returns 'review' rather than 'fail'."""
    tokens = norm_tokens(canonical_container)
    if not tokens:
        return 'review'
    ct = set(norm_tokens(citation_text))
    title_tokens = set(norm_tokens(canonical_title))
    distinct = [t for t in tokens if t not in title_tokens]
    raw_present = sum(1 for t in tokens if t in ct)

    if raw_present == len(tokens):
        if not distinct:
            return 'review'  # container fully shadowed by title words
        return 'pass' if any(t in ct for t in distinct) else 'review'

    if not distinct:
        return 'review'
    distinct_present = sum(1 for t in distinct if t in ct)
    if distinct_present == 0:
        text_l = re.sub(r'[^a-z0-9 ]', ' ', citation_text.lower())
        abbrev_hits = sum(1 for t in distinct
                          if re.search(r'\b' + re.escape(t[:4]), text_l))
        return 'review' if abbrev_hits == len(distinct) else 'fail'
    return 'review'

def year_check(citation_text, canonical_year):
    if not canonical_year:
        return 'review'
    years = {int(y) for y in re.findall(r'\b(19\d{2}|20\d{2})\b', citation_text)}
    if not years:
        return 'review'
    if canonical_year in years:
        return 'pass'
    if any(abs(y - canonical_year) <= 1 for y in years):
        return 'review'
    return 'fail'

def author_check(citation_text, authors):
    """Is the first author's family name present in the citation text?"""
    if not authors:
        return 'review'
    family = (authors[0].get('family') or '').strip()
    if not family:
        return 'review'
    return 'pass' if re.search(r'\b' + re.escape(family.lower()) + r'\b',
                               citation_text.lower()) else 'fail'

def format_citation(msg):
    """Render canonical CrossRef metadata as plain reference text."""
    authors = msg.get('author') or []
    names = ', '.join(
        f"{a.get('family', '')}, {a.get('given', '')}".strip(', ')
        for a in authors[:6]) + (' et al.' if len(authors) > 6 else '')
    year = extract_year(msg) or 'n.d.'
    title = (msg.get('title') or [''])[0]
    container = (msg.get('container-title') or [''])[0]
    vol = msg.get('volume', '')
    issue = msg.get('issue', '')
    pages = msg.get('page', '')
    doi = msg.get('DOI', '')
    parts = [f"{names}. {year}. \"{title}.\""]
    if container:
        loc = container + (f" {vol}" if vol else '') + (f"({issue})" if issue else '')
        loc += f": {pages}" if pages else ''
        parts.append(loc + '.')
    parts.append(f"https://doi.org/{doi}")
    return ' '.join(p for p in parts if p)

def extract_year(msg):
    for field in ('published-print', 'published-online', 'issued'):
        dp = (msg.get(field) or {}).get('date-parts')
        if dp and dp[0] and dp[0][0]:
            return int(dp[0][0])
    return None

def build_bibtex(msg):
    """Minimal BibTeX entry from canonical metadata."""
    authors = msg.get('author') or []
    family = (authors[0].get('family') if authors else 'anon') or 'anon'
    year = extract_year(msg) or 'nd'
    key = re.sub(r'[^a-z0-9]', '', f"{family}{year}".lower()) or 'ref'
    kind = {'journal-article': 'article', 'book': 'book',
            'book-chapter': 'incollection',
            'proceedings-article': 'inproceedings'}.get(msg.get('type'), 'misc')
    fields = {
        'author': ' and '.join(f"{a.get('family', '')}, {a.get('given', '')}"
                               for a in authors if a.get('family')),
        'title': (msg.get('title') or [''])[0],
        'journal': (msg.get('container-title') or [''])[0],
        'year': str(year), 'volume': msg.get('volume', ''),
        'number': msg.get('issue', ''), 'pages': msg.get('page', ''),
        'doi': msg.get('DOI', ''),
    }
    body = ',\n'.join(f"  {k} = {{{v}}}" for k, v in fields.items() if v)
    return f"@{kind}{{{key},\n{body}\n}}"

print()
print(f"✓ Environment: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print("\U0001f50e Ready to configure!")


## Configuration

Set your email for CrossRef's polite pool — identified requests get faster, more reliable service — and adjust request pacing if needed.

In [ ]:
# Configuration

class VerifierConfig:
    EMAIL = ''
    REQUEST_DELAY = 1.0
    ROWS_PER_PAGE = 100
    MAX_PAGES = 50
    CHECK_RETRACTIONS = True
    PROJECT_NAME = ''

config = VerifierConfig()
API_BASE = "https://api.crossref.org"

def api_headers():
    ua = "AIAnthropologyToolkit-CrossRefVerifier/1.0"
    if config.EMAIL:
        ua += f" (mailto:{config.EMAIL})"
    return {"User-Agent": ua}

def api_params(extra=None):
    p = dict(extra or {})
    if config.EMAIL:
        p['mailto'] = config.EMAIL
    return p

def crossref_get(path, params=None):
    """GET with polite identification, pacing, and honest errors."""
    r = requests.get(f"{API_BASE}{path}", params=api_params(params),
                     headers=api_headers(), timeout=60)
    if r.status_code == 404:
        return None
    r.raise_for_status()
    return r.json().get('message')

style = {'description_width': '160px'}
layout = widgets.Layout(width='500px')

header_html = '<div style="background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;">'
header_html += '<h2 style="color: #274C77; margin-top: 0;">\U0001f50e CrossRef Reference Verifier</h2>'
header_html += '<p><strong>Welcome!</strong> This notebook verifies reference lists against canonical CrossRef metadata and queries journals and authors directly.</p>'
header_html += '<ol>'
header_html += '<li><strong>Configure:</strong> Enter your email for polite-pool access</li>'
header_html += '<li><strong>Verify:</strong> Paste a reference list and run the report</li>'
header_html += '<li><strong>Query:</strong> Pull journal or author records if needed</li>'
header_html += '<li><strong>Export:</strong> Download reports and data</li>'
header_html += '</ol></div>'

email_input = widgets.Text(value='', placeholder='you@example.edu',
                           description='Email (polite pool):', layout=layout, style=style)
email_help = widgets.HTML('<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
    'Optional but recommended: CrossRef gives identified requests priority service. '
    'Your email goes only to CrossRef in the request header.</p>')

delay_input = widgets.FloatSlider(value=1.0, min=0.5, max=5.0, step=0.5,
                                  description='Request delay (s):', layout=layout, style=style)

retraction_toggle = widgets.Checkbox(value=True, description='Check for retraction/correction notices',
                                     style={'description_width': 'auto'})

project_input = widgets.Text(value='', placeholder='e.g., Dissertation References',
                             description='Project name:', layout=layout, style=style)

save_btn = widgets.Button(description='Save Configuration',
                          style={'button_color': '#6096BA'},
                          layout=widgets.Layout(width='200px', margin='20px 0'))
config_out = widgets.Output()

def on_save_config(_):
    config.EMAIL = email_input.value.strip()
    config.REQUEST_DELAY = delay_input.value
    config.CHECK_RETRACTIONS = retraction_toggle.value
    config.PROJECT_NAME = project_input.value.strip()
    with config_out:
        clear_output()
        print("Checking CrossRef API...")
        try:
            msg = crossref_get('/works', {'rows': 0, 'query': 'test'})
            total = msg.get('total-results', 0) if msg else 0
            print(f"✓ CrossRef reachable — {total:,} works indexed")
            print(f"✓ Polite pool: {'enabled (' + config.EMAIL + ')' if config.EMAIL else 'not set (anonymous pool)'}")
            print(f"✓ Retraction checks: {'on' if config.CHECK_RETRACTIONS else 'off'}")
            print()
            print("✓ Configuration saved! Proceed to Verify References.")
        except Exception as e:
            print(f"❌ CrossRef not reachable: {e}")

save_btn.on_click(on_save_config)
display(widgets.VBox([widgets.HTML(header_html), email_input, email_help,
                      delay_input, retraction_toggle, project_input,
                      save_btn, config_out]))


## Verify References

Paste a reference section (or upload a .txt/.docx file), then run verification. Each entry is resolved against CrossRef and compared field by field; uncertain cases are flagged for your review rather than auto-judged.

In [ ]:
# Verify References

verification_results = []

refs_input = widgets.Textarea(
    value='', placeholder='Paste your reference list here — one entry per line or separated by blank lines...',
    layout=widgets.Layout(width='95%', height='220px'))

ref_uploader = widgets.FileUpload(accept='.txt,.docx', multiple=False,
                                  description='...or upload file',
                                  layout=widgets.Layout(width='220px'))

def on_ref_upload(change):
    for f in ref_uploader.value:
        name = f['name'] if isinstance(f, dict) else f.name
        content = bytes(f['content'] if isinstance(f, dict) else f.content)
        if name.lower().endswith('.docx'):
            import io
            d = docx.Document(io.BytesIO(content))
            refs_input.value = '\n'.join(p.text for p in d.paragraphs if p.text.strip())
        else:
            try:
                refs_input.value = content.decode('utf-8')
            except UnicodeDecodeError:
                refs_input.value = content.decode('latin-1')

ref_uploader.observe(on_ref_upload, names='value')

def check_retraction(doi):
    """Return list of notice types (retraction, correction, ...) updating this DOI."""
    try:
        msg = crossref_get('/works', {'filter': f'updates:{doi}', 'rows': 5,
                                      'select': 'DOI,type,update-to,title'})
        notices = []
        for item in (msg or {}).get('items', []):
            for upd in item.get('update-to', []):
                if clean_doi(upd.get('DOI', '')) == doi.lower():
                    notices.append(upd.get('type', 'update'))
        return sorted(set(notices))
    except Exception:
        return []

def verify_entry(entry):
    """Verify one reference entry against CrossRef. Returns a result dict."""
    doi = clean_doi(entry)
    result = {'reference_text': entry[:400], 'doi': doi or '', 'verdict': '',
              'issues': '', 'canonical_citation': '', 'retraction': '',
              'match_confidence': ''}

    msg = None
    if doi:
        msg = crossref_get(f'/works/{doi}')
        if msg is None:
            result['verdict'] = 'DOI INVALID'
            result['issues'] = 'DOI does not resolve in CrossRef'
            return result
    else:
        found = crossref_get('/works', {'query.bibliographic': entry[:300], 'rows': 2,
                                        'select': 'DOI,title,author,issued,container-title,score,volume,issue,page,type,published-print,published-online'})
        items = (found or {}).get('items', [])
        if not items:
            result['verdict'] = 'NO MATCH'
            result['issues'] = 'No DOI in text and no plausible CrossRef match — verify this reference exists'
            return result
        msg = items[0]
        result['doi'] = msg.get('DOI', '')
        result['match_confidence'] = f"bibliographic match (score {msg.get('score', 0):.0f}) — confirm manually"

    checks = {
        'title': 'pass' if title_similarity(entry, (msg.get('title') or [''])[0]) >= 0.6
                 else ('review' if title_similarity(entry, (msg.get('title') or [''])[0]) >= 0.35 else 'fail'),
        'first author': author_check(entry, msg.get('author')),
        'year': year_check(entry, extract_year(msg)),
        'journal': container_check(entry, (msg.get('container-title') or [''])[0],
                                   (msg.get('title') or [''])[0]),
    }
    fails = [k for k, v in checks.items() if v == 'fail']
    reviews = [k for k, v in checks.items() if v == 'review']

    # A no-DOI entry whose best bibliographic candidate barely corresponds is
    # not a mismatch — it is a reference CrossRef cannot substantiate.
    if result['match_confidence'] and len(fails) >= 2:
        result['verdict'] = 'NO MATCH'
        result['issues'] = ('no DOI given and the closest CrossRef candidate does not '
                            'correspond — verify this reference exists')
        result['canonical_citation'] = ''
        return result

    detail = []
    if 'journal' in fails:
        detail.append(f"text journal differs from canonical '{(msg.get('container-title') or [''])[0]}'")
    if 'year' in fails:
        detail.append(f"text year differs from canonical {extract_year(msg)}")
    if 'title' in fails:
        detail.append('title does not match canonical record')
    if 'first author' in fails:
        fam = (msg.get('author') or [{}])[0].get('family', '?')
        detail.append(f"canonical first author '{fam}' not found in text")

    if fails:
        result['verdict'] = 'MISMATCH'
        result['issues'] = '; '.join(detail)
    elif reviews or result['match_confidence']:
        result['verdict'] = 'REVIEW'
        result['issues'] = 'check: ' + ', '.join(reviews) if reviews else ''
    else:
        result['verdict'] = 'OK'

    result['canonical_citation'] = format_citation(msg)
    result['_message'] = msg

    if config.CHECK_RETRACTIONS and result['doi']:
        time.sleep(config.REQUEST_DELAY / 2)
        notices = check_retraction(result['doi'])
        if notices:
            result['retraction'] = ', '.join(notices).upper()
            if any('retract' in n for n in notices):
                result['verdict'] = 'RETRACTED'
    return result

verify_btn = widgets.Button(description='Verify References',
                            style={'button_color': '#6096BA'},
                            layout=widgets.Layout(width='200px', margin='10px 0'))
verify_progress = widgets.HTML(value='')
verify_out = widgets.Output()

VERDICT_COLORS = {'OK': '#2E7D32', 'REVIEW': '#B8860B', 'MISMATCH': '#C62828',
                  'DOI INVALID': '#C62828', 'NO MATCH': '#C62828', 'RETRACTED': '#8B0000'}

def on_verify(_):
    global verification_results
    verify_out.clear_output()
    entries = split_reference_entries(refs_input.value)
    if not entries:
        with verify_out:
            print("⚠️ No references found — paste a reference list above.")
        return
    verification_results = []
    with verify_out:
        print(f"\U0001f50e Verifying {len(entries)} reference(s) against CrossRef...")
        print()
    for i, entry in enumerate(entries):
        verify_progress.value = (f'<span style="color: #274C77;">Checking {i + 1}/{len(entries)} — '
                                 f'{entry[:70]}...</span>')
        try:
            verification_results.append(verify_entry(entry))
        except Exception as e:
            verification_results.append({'reference_text': entry[:400], 'doi': '',
                                         'verdict': 'ERROR', 'issues': str(e)[:200],
                                         'canonical_citation': '', 'retraction': '',
                                         'match_confidence': ''})
        time.sleep(config.REQUEST_DELAY)
    verify_progress.value = ''

    with verify_out:
        counts = {}
        for r in verification_results:
            counts[r['verdict']] = counts.get(r['verdict'], 0) + 1
        summary = ' | '.join(f"{v}: {c}" for v, c in sorted(counts.items()))
        print(f"✓ Done — {summary}")
        print()
        rows_html = ''
        for i, r in enumerate(verification_results, 1):
            color = VERDICT_COLORS.get(r['verdict'], '#274C77')
            rows_html += '<tr>'
            rows_html += f'<td style="padding:6px; color:#8B8C89;">{i}</td>'
            rows_html += f'<td style="padding:6px; max-width:420px;">{r["reference_text"][:200]}</td>'
            rows_html += f'<td style="padding:6px; font-weight:bold; color:{color};">{r["verdict"]}</td>'
            issue_bits = '; '.join(x for x in (r['issues'], r['match_confidence'], r['retraction']) if x)
            rows_html += f'<td style="padding:6px; max-width:320px; color:#274C77;">{issue_bits}</td>'
            rows_html += '</tr>'
        table = ('<table style="border-collapse:collapse; font-size:13px;">'
                 '<tr style="background:#274C77; color:white;">'
                 '<th style="padding:6px;">#</th><th style="padding:6px;">Reference</th>'
                 '<th style="padding:6px;">Verdict</th><th style="padding:6px;">Notes</th></tr>'
                 + rows_html + '</table>')
        display(HTML(table))
        flagged = [r for r in verification_results if r['verdict'] != 'OK']
        if flagged:
            print()
            print("Canonical citations for flagged entries:")
            for r in flagged:
                if r['canonical_citation']:
                    print(f"  • {r['canonical_citation']}")

verify_btn.on_click(on_verify)
display(widgets.VBox([refs_input, ref_uploader, verify_btn, verify_progress, verify_out]))


## Journal Query

Retrieve works from any journal by ISSN, with optional date filtering. Separate multiple ISSNs with commas to query several journals in one run.

In [ ]:
# Journal Query

journal_df = None

issn_input = widgets.Text(value='', placeholder='e.g., 0002-7294, 1548-1433',
                          description='ISSN(s):', layout=layout, style=style)
jr_from = widgets.Text(value='', placeholder='YYYY-MM-DD (optional)',
                       description='From date:', layout=layout, style=style)
jr_until = widgets.Text(value='', placeholder='YYYY-MM-DD (optional)',
                        description='Until date:', layout=layout, style=style)
jr_max = widgets.IntSlider(value=500, min=100, max=5000, step=100,
                           description='Max records:', layout=layout, style=style)

journal_btn = widgets.Button(description='Query Journal(s)',
                             style={'button_color': '#6096BA'},
                             layout=widgets.Layout(width='200px', margin='10px 0'))
journal_progress = widgets.HTML(value='')
journal_out = widgets.Output()

SELECT_FIELDS = ('DOI,title,author,issued,published-print,published-online,'
                 'container-title,volume,issue,page,type,is-referenced-by-count,ISSN')

def works_to_rows(items, source_label):
    rows = []
    for msg in items:
        authors = msg.get('author') or []
        rows.append({
            'doi': msg.get('DOI', ''),
            'title': normalize_text((msg.get('title') or [''])[0]),
            'authors': '; '.join(f"{a.get('family', '')}, {a.get('given', '')}".strip(', ')
                                 for a in authors),
            'year': extract_year(msg),
            'journal': (msg.get('container-title') or [''])[0],
            'volume': msg.get('volume', ''), 'issue': msg.get('issue', ''),
            'pages': msg.get('page', ''), 'type': msg.get('type', ''),
            'cited_by': msg.get('is-referenced-by-count', 0),
            'source_query': source_label,
        })
    return rows

def paged_works(path, params, max_records, progress, label):
    """Cursor-paginate a works endpoint up to max_records."""
    rows, cursor = [], '*'
    while len(rows) < max_records:
        page = crossref_get(path, {**params, 'rows': min(config.ROWS_PER_PAGE, max_records - len(rows)),
                                   'cursor': cursor, 'select': SELECT_FIELDS})
        items = (page or {}).get('items', [])
        if not items:
            break
        rows.extend(works_to_rows(items, label))
        progress.value = f'<span style="color: #274C77;">{label}: {len(rows)} records...</span>'
        cursor = (page or {}).get('next-cursor')
        if not cursor:
            break
        time.sleep(config.REQUEST_DELAY)
    return rows

def on_journal_query(_):
    global journal_df
    journal_out.clear_output()
    issns = [s.strip() for s in issn_input.value.split(',') if s.strip()]
    if not issns:
        with journal_out:
            print("⚠️ Enter at least one ISSN.")
        return
    filters = []
    if jr_from.value.strip():
        filters.append(f"from-pub-date:{jr_from.value.strip()}")
    if jr_until.value.strip():
        filters.append(f"until-pub-date:{jr_until.value.strip()}")
    all_rows = []
    with journal_out:
        for issn in issns:
            try:
                params = {'filter': ','.join(filters)} if filters else {}
                rows = paged_works(f'/journals/{issn}/works', params,
                                   jr_max.value, journal_progress, issn)
                print(f"✓ {issn}: {len(rows)} records")
                all_rows.extend(rows)
            except Exception as e:
                print(f"❌ {issn}: {e}")
        journal_progress.value = ''
        if all_rows:
            journal_df = pd.DataFrame(all_rows)
            print()
            print(f"✓ Total: {len(journal_df)} records from {len(issns)} journal(s)")
            display(journal_df.head(10))

journal_btn.on_click(on_journal_query)
display(widgets.VBox([issn_input, jr_from, jr_until, jr_max,
                      journal_btn, journal_progress, journal_out]))


## Author Query

Retrieve all works CrossRef links to an author's ORCID iD. Accepts the bare iD or the full orcid.org URL.

In [ ]:
# Author Query

author_df = None

orcid_input = widgets.Text(value='', placeholder='e.g., 0000-0002-1825-0097',
                           description='ORCID iD:', layout=layout, style=style)
au_max = widgets.IntSlider(value=500, min=100, max=2000, step=100,
                           description='Max records:', layout=layout, style=style)

author_btn = widgets.Button(description='Query Author',
                            style={'button_color': '#6096BA'},
                            layout=widgets.Layout(width='200px', margin='10px 0'))
author_progress = widgets.HTML(value='')
author_out = widgets.Output()

def on_author_query(_):
    global author_df
    author_out.clear_output()
    orcid = clean_orcid(orcid_input.value)
    if not orcid:
        with author_out:
            print("⚠️ Enter a valid ORCID iD (0000-0000-0000-0000).")
        return
    with author_out:
        try:
            rows = paged_works('/works', {'filter': f'orcid:{orcid}'},
                               au_max.value, author_progress, orcid)
            author_progress.value = ''
            if rows:
                author_df = pd.DataFrame(rows)
                print(f"✓ {len(author_df)} works linked to {orcid}")
                display(author_df.head(10))
            else:
                print(f"No works found for {orcid}. CrossRef only links works where "
                      "the publisher deposited the ORCID — coverage is partial.")
        except Exception as e:
            print(f"❌ Query failed: {e}")

author_btn.on_click(on_author_query)
display(widgets.VBox([orcid_input, au_max, author_btn, author_progress, author_out]))


## Export

Download the verification report or query results. The verification Excel report includes a summary sheet; BibTeX exports canonical entries for every verified reference.

In [ ]:
# Export

export_choice = widgets.Dropdown(
    options=[('Verification report — CSV', 'verify_csv'),
             ('Verification report — Excel (styled)', 'verify_excel'),
             ('Verification report — JSON', 'verify_json'),
             ('Canonical references — BibTeX', 'verify_bibtex'),
             ('Journal query results — CSV', 'journal_csv'),
             ('Journal query results — Excel (styled)', 'journal_excel'),
             ('Author query results — CSV', 'author_csv'),
             ('Author query results — Excel (styled)', 'author_excel')],
    value='verify_csv', description='Export:', layout=layout, style=style)

export_btn = widgets.Button(description='Export',
                            style={'button_color': '#6096BA'},
                            layout=widgets.Layout(width='200px', margin='10px 0'))
export_out = widgets.Output()

def _verify_df():
    return pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')}
                         for r in verification_results])

def on_export(_):
    export_out.clear_output()
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    slug = make_slug(config.PROJECT_NAME) if config.PROJECT_NAME else 'crossref'
    choice = export_choice.value
    with export_out:
        try:
            if choice.startswith('verify') and not verification_results:
                print("⚠️ Run Verify References first.")
                return
            if choice.startswith('journal') and journal_df is None:
                print("⚠️ Run the Journal Query first.")
                return
            if choice.startswith('author') and author_df is None:
                print("⚠️ Run the Author Query first.")
                return

            if choice == 'verify_csv':
                path = f'verification_{slug}_{timestamp}.csv'
                _verify_df().to_csv(path, index=False)
            elif choice == 'verify_excel':
                path = f'verification_{slug}_{timestamp}.xlsx'
                df = _verify_df()
                counts = df['verdict'].value_counts().rename_axis('Verdict').reset_index(name='Count')
                with pd.ExcelWriter(path, engine='openpyxl') as w:
                    counts.to_excel(w, sheet_name='Summary', index=False)
                    df.to_excel(w, sheet_name='Verification', index=False)
                style_excel(path)
            elif choice == 'verify_json':
                path = f'verification_{slug}_{timestamp}.json'
                with open(path, 'w') as f:
                    json.dump({'project': config.PROJECT_NAME,
                               'verified_at': datetime.now().isoformat(),
                               'results': [{k: v for k, v in r.items() if not k.startswith('_')}
                                           for r in verification_results]}, f, indent=2)
            elif choice == 'verify_bibtex':
                path = f'references_{slug}_{timestamp}.bib'
                entries = [build_bibtex(r['_message']) for r in verification_results
                           if r.get('_message')]
                with open(path, 'w') as f:
                    f.write('\n\n'.join(entries))
            elif choice in ('journal_csv', 'author_csv'):
                df = journal_df if choice.startswith('journal') else author_df
                path = f'{choice.split("_")[0]}_works_{slug}_{timestamp}.csv'
                df.to_csv(path, index=False)
            else:
                df = journal_df if choice.startswith('journal') else author_df
                path = f'{choice.split("_")[0]}_works_{slug}_{timestamp}.xlsx'
                with pd.ExcelWriter(path, engine='openpyxl') as w:
                    df.to_excel(w, sheet_name='Works', index=False)
                style_excel(path)

            print(f"✓ Saved: {path}")
            if IN_COLAB:
                colab_files.download(path)
        except Exception as e:
            print(f"❌ Export error: {type(e).__name__}: {e}")

export_btn.on_click(on_export)
display(widgets.VBox([export_choice, export_btn, export_out]))
